# Tests for `encryp.ipynb`

Every code cell tagged **`test`** is a piece of one pytest module. The runner
at the bottom stitches them together and hands them to pytest, so this stays
an ordinary test suite rather than a pile of bare asserts.

## How to run

- **In Jupyter** — Run All. The last cell prints the pytest report.
- **Only some tests** — call the runner directly, e.g.
  `run_tests("-k", "round_trip", "-v")`.
- **Headless / CI** — `jupyter nbconvert --execute --to notebook --stdout
  ver_1/test_encryp.ipynb`, or import `build_test_module()` and write the
  module out yourself.

## What is covered

| Area | Guards against |
| --- | --- |
| Round trip | Ciphertext containing `_` being re-split at the wrong place, which used to make ~3% of tokens permanently undecryptable |
| Injectivity | Padding mapping two different suffixes onto one token |
| Issuance | Partial commits, tokens issued without a vault record, one token meaning two plaintexts |
| Detokenization | Cross-case reads, invented tokens, key rotation, refused attempts going unaudited |
| Configuration | Missing env vars, non-idempotent schema |

## Notes

- Tests reach into `namespace`, which is the executed `library` cell of
  `encryp.ipynb`. Editing that cell and re-running here picks up the change.
- The `cipher` fixture pins a fixed key, so a few tests name specific values
  (`[HOST_54]`, ...) that produce `_` in their ciphertext under that key.
  Change the key and those particular regression cases lose their point,
  though `test_round_trip_holds_across_the_whole_suffix_space` still covers
  the property for any key.

## Loading the library notebook

In [ ]:
import json
import os
from collections import Counter
from pathlib import Path

import pytest


def locate_notebook(name: str = "encryp.ipynb") -> Path:
    """Find the library notebook from a notebook, a script, or a temp module.

    Inside Jupyter there is no __file__, and the runner cell below assembles
    these cells into a module outside this directory, so the path is resolved
    at run time instead of being assumed.
    """
    override = os.environ.get("ENCRYP_NOTEBOOK")
    if override:
        return Path(override)
    roots = [Path(__file__).parent] if "__file__" in globals() else []
    roots += [Path.cwd(), Path.cwd() / "ver_1"]
    for root in roots:
        candidate = root / name
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f"could not locate {name} from {Path.cwd()}")


NOTEBOOK = locate_notebook()


def load_library_cells() -> str:
    """Return the source of the notebook cells tagged 'library'.

    Selecting by tag rather than by index keeps the tests working when cells
    are reordered, and keeps scratch cells out of the test run.
    """
    with NOTEBOOK.open(encoding="utf-8") as notebook_file:
        notebook = json.load(notebook_file)
    code_cells = [cell for cell in notebook["cells"] if cell["cell_type"] == "code"]
    tagged = [cell for cell in code_cells if "library" in cell["metadata"].get("tags", [])]
    if not tagged:
        raise AssertionError("encryp.ipynb has no code cell tagged 'library'")
    return "\n".join("".join(cell["source"]) for cell in tagged)


namespace = {}
exec(compile(load_library_cells(), str(NOTEBOOK), "exec"), namespace)

ALPHABET = namespace["ALPHABET"]
PAD_CHAR = namespace["PAD_CHAR"]
KeyRing = namespace["KeyRing"]
TokenCollisionError = namespace["TokenCollisionError"]
TokenHallucinationError = namespace["TokenHallucinationError"]
TokenLengthError = namespace["TokenLengthError"]
UnknownKeyVersionError = namespace["UnknownKeyVersionError"]
VersionedCipher = namespace["VersionedCipher"]
detokenize_log = namespace["detokenize_log"]
ensure_schema = namespace["ensure_schema"]
key_versions_in_use = namespace["key_versions_in_use"]
load_key_ring = namespace["load_key_ring"]
lookup_issued_token = namespace["lookup_issued_token"]
record_issued_token = namespace["record_issued_token"]
safe_decrypt = namespace["safe_decrypt"]
tokenize_field = namespace["tokenize_field"]
tokenize_log = namespace["tokenize_log"]
walk_and_tokenize = namespace["walk_and_tokenize"]

KEY_V1 = "F3D2FE0E66B2AA56806944AA2770D53A"
TWEAK_V1 = "6A81B1DC1D04B6CA"
KEY_V2 = "2DE79D232DF5585D68CE47882AE256D6"
TWEAK_V2 = "9A768A92F60E12D8"

## A fake database

In [ ]:
class FakeCursor:
    """In-memory stand-in for a psycopg cursor, faithful about rowcount."""

    def __init__(self, connection):
        self.connection = connection
        self.row = None
        self.rows = []
        self.rowcount = -1

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc_value, traceback):
        return False

    def execute(self, query, params=()):
        normalized_query = " ".join(query.split()).upper()
        self.connection.executed.append((normalized_query, params))
        self.row = None
        self.rows = []
        self.rowcount = -1

        if normalized_query.startswith("CREATE SCHEMA"):
            self.connection.schema_ready = True
        elif normalized_query.startswith("SELECT PREFIX"):
            token, case_id = params
            self.row = self.connection.issued.get((token, case_id))
        elif normalized_query.startswith("SELECT KEY_VERSION"):
            counts = Counter(
                key_version
                for (_, case_id), (_, _, key_version) in self.connection.issued.items()
                if not params or case_id == params[0]
            )
            self.rows = sorted(counts.items())
        elif normalized_query.startswith("INSERT INTO VAULT_SCHEMA.ISSUED_TOKENS"):
            assert "ON CONFLICT (TOKEN, CASE_ID) DO NOTHING" in normalized_query
            token, prefix, suffix_len, case_id, key_version = params
            if (token, case_id) in self.connection.issued:
                self.rowcount = 0
            else:
                self.connection.issued[(token, case_id)] = (prefix, suffix_len, key_version)
                self.rowcount = 1
        elif normalized_query.startswith("INSERT INTO METADATA_SCHEMA.DETOKENIZE_LOG"):
            self.connection.audit_log.append(params)
            self.rowcount = 1
        else:
            raise AssertionError(f"unexpected query: {normalized_query}")

    def fetchone(self):
        return self.row

    def fetchall(self):
        return self.rows


class FakeConnection:
    def __init__(self):
        self.issued = {}
        self.audit_log = []
        self.executed = []
        self.commit_count = 0
        self.schema_ready = False

    def cursor(self):
        return FakeCursor(self)

    def commit(self):
        self.commit_count += 1

## Fixtures

In [ ]:
@pytest.fixture
def env(monkeypatch):
    """A hermetic FF3 environment holding only the v1 key."""
    for name in list(os.environ):
        if name.startswith("FF3_"):
            monkeypatch.delenv(name, raising=False)
    monkeypatch.setenv("FF3_KEY", KEY_V1)
    monkeypatch.setenv("FF3_TWEAK", TWEAK_V1)
    return monkeypatch


@pytest.fixture
def key_ring(env):
    return load_key_ring()


def rotate_to_v2(env):
    """Perform a rotation: add v2, keep v1 loaded, make v2 active."""
    env.delenv("FF3_KEY", raising=False)
    env.delenv("FF3_TWEAK", raising=False)
    env.setenv("FF3_KEY_V1", KEY_V1)
    env.setenv("FF3_TWEAK_V1", TWEAK_V1)
    env.setenv("FF3_KEY_V2", KEY_V2)
    env.setenv("FF3_TWEAK_V2", TWEAK_V2)
    env.setenv("FF3_ACTIVE_KEY_VERSION", "v2")
    return load_key_ring()


def retire_v1(env):
    """Take v1 off the ring entirely, leaving only v2 loaded."""
    env.delenv("FF3_KEY", raising=False)
    env.delenv("FF3_TWEAK", raising=False)
    env.delenv("FF3_KEY_V1", raising=False)
    env.delenv("FF3_TWEAK_V1", raising=False)
    env.setenv("FF3_KEY_V2", KEY_V2)
    env.setenv("FF3_TWEAK_V2", TWEAK_V2)
    env.setenv("FF3_ACTIVE_KEY_VERSION", "v2")
    return load_key_ring()


@pytest.fixture
def rotated(env):
    return rotate_to_v2(env)


@pytest.fixture
def connection():
    return FakeConnection()


def round_trip(connection, key_ring, original, case_id="INC-001"):
    token = tokenize_log(original, key_ring, connection, case_id)
    return token, safe_decrypt(connection, key_ring, token, case_id, "analyst")

## Round trip

In [ ]:
def test_round_trip_and_prefix_is_preserved(connection, key_ring):
    token, restored = round_trip(connection, key_ring, "[HOST_01]")

    assert token.startswith("[HOST_")
    assert restored == "[HOST_01]"
    assert connection.audit_log == [(token, "INC-001", "analyst", "success")]


@pytest.mark.parametrize(
    "original",
    ["[HOST_54]", "[HOST_88]", "[HOST_110]", "[HOST_125]", "[HOST_212]"],
)
def test_round_trip_when_ciphertext_contains_the_pad_character(
    connection, key_ring, original
):
    """Regression: these encrypt to ciphertext containing '_' under the test key.

    Re-parsing such a token with PLACEHOLDER_RE splits the prefix in a
    different place than issuance did, which used to make them permanently
    undecryptable.
    """
    token, restored = round_trip(connection, key_ring, original)

    assert PAD_CHAR in token[len("[HOST_") :]
    assert restored == original


def test_round_trip_holds_across_the_whole_suffix_space(connection, key_ring):
    """Property check: nothing that goes in may fail to come back out."""
    failures = []
    for prefix in ("HOST", "INT_IP", "USER", "LOG_COLLECTOR"):
        for index in range(150):
            original = f"[{prefix}_{index:02d}]"
            try:
                _, restored = round_trip(connection, key_ring, original, f"CASE-{index}")
            except Exception as error:  # noqa: BLE001 - reported below
                failures.append((original, f"{type(error).__name__}: {error}"))
                continue
            if restored != original:
                failures.append((original, f"restored as {restored}"))

    assert not failures, f"{len(failures)} of 600 values did not round-trip: {failures[:5]}"


def test_distinct_values_never_share_a_token(key_ring):
    """Padding must not map two different suffixes onto one ciphertext."""
    seen = {}
    for index in range(300):
        original = f"[HOST_{index}]"
        token = tokenize_field(key_ring, original)
        assert token not in seen, f"{original} and {seen[token]} both produced {token}"
        seen[token] = original


@pytest.mark.parametrize(
    "short, underscored",
    [("[HOST_01]", "[HOST__01]"), ("[USER_A]", "[USER__A]"), ("[HOST_1]", "[HOST___1]")],
)
def test_padding_is_never_confused_with_a_literal_underscore(key_ring, short, underscored):
    """The exact collision the old suffix pattern allowed.

    "01" is padded to "__01" before encryption. While a suffix may itself
    contain '_', the literal suffix "_01" pads to the same "__01" and both
    values collapse onto one token.
    """
    assert tokenize_field(key_ring, short) != tokenize_field(key_ring, underscored)


def test_short_suffix_is_padded_only_inside_cipher(key_ring):
    token = tokenize_field(key_ring, "[HOST_01]")

    assert token.startswith("[HOST_")
    assert len(token.removeprefix("[HOST_").removesuffix("]")) >= key_ring.active.minLen


def test_embedded_placeholder_preserves_surrounding_text(key_ring):
    result = tokenize_field(key_ring, "Connection from [INT_IP_01] refused")

    assert result.startswith("Connection from [INT_IP_")
    assert result.endswith("] refused")


def test_suffix_longer_than_the_cipher_maximum_is_rejected(key_ring):
    too_long = "[HOST_" + "A" * (key_ring.active.maxLen + 1) + "]"

    with pytest.raises(TokenLengthError, match="exceeds the cipher maximum"):
        tokenize_field(key_ring, too_long)


def test_bracketed_text_that_is_not_a_placeholder_passes_through(key_ring):
    """Documented behaviour: only well-formed placeholders are tokenized."""
    for value in ["logs[6124]", "[HOST__01]", "[1003]", "[]"]:
        assert tokenize_field(key_ring, value) == value

## Issuance

In [ ]:
def test_walk_records_all_issued_tokens_for_case(connection, key_ring):
    result = walk_and_tokenize(
        {"message": "from [INT_IP_01] as [USER_01]", "nested": ["[HOST_01]"]},
        key_ring,
        connection,
        "INC-001",
    )

    assert result["message"] != "from [INT_IP_01] as [USER_01]"
    assert len(connection.issued) == 3
    assert all(case_id == "INC-001" for _, case_id in connection.issued)


def test_tokenize_log_commits_once_per_log_not_once_per_token(connection, key_ring):
    tokenize_log(
        {"a": "[HOST_01]", "b": "[USER_02]", "c": ["[INT_IP_03]", "[AGENT_04]"]},
        key_ring,
        connection,
        "INC-001",
    )

    assert len(connection.issued) == 4
    assert connection.commit_count == 1


def test_issuing_without_a_case_id_is_refused(connection, key_ring):
    with pytest.raises(ValueError, match="must be given together"):
        tokenize_field(key_ring, "[HOST_01]", connection, None)

    with pytest.raises(ValueError, match="must be given together"):
        walk_and_tokenize({"a": "[HOST_01]"}, key_ring, connection, None)

    assert connection.issued == {}


def test_repeated_value_is_recorded_once(connection, key_ring):
    tokenize_log({"a": "[HOST_01]", "b": "[HOST_01]"}, key_ring, connection, "INC-001")

    assert len(connection.issued) == 1


def test_one_token_mapping_to_two_plaintexts_is_refused(connection):
    record_issued_token(connection, "[HOST_AB_CD]", "HOST", 2, "INC-001", "v1")

    with pytest.raises(TokenCollisionError):
        record_issued_token(connection, "[HOST_AB_CD]", "HOST_AB", 2, "INC-001", "v1")

## Detokenization

In [ ]:
def test_unissued_token_is_rejected_before_decrypt(connection, key_ring):
    with pytest.raises(TokenHallucinationError) as error:
        safe_decrypt(connection, key_ring, "[HOST_01]", "INC-001", "analyst")

    assert error.value.token == "[HOST_01]"
    assert error.value.case_id == "INC-001"


def test_refused_detokenization_is_still_audited(connection, key_ring):
    with pytest.raises(TokenHallucinationError):
        safe_decrypt(connection, key_ring, "[HOST_99]", "INC-001", "mallory")

    assert connection.audit_log == [("[HOST_99]", "INC-001", "mallory", "rejected")]
    assert connection.commit_count == 1, "the audit row must be committed, not lost"


def test_token_is_isolated_by_case_id(connection, key_ring):
    token = tokenize_log("[HOST_01]", key_ring, connection, "INC-001")

    with pytest.raises(TokenHallucinationError):
        safe_decrypt(connection, key_ring, token, "INC-002", "analyst")


def test_whole_document_round_trips_and_leaves_non_tokens_alone(connection, key_ring):
    original = {
        "metadata": {"log_source": "[LOG_COLLECTOR_01] logs[6124]", "event_code": "1003"},
        "device": {"hostname": "[HOST_01]", "ip": "[INT_IP_01]", "type_id": 2},
        "evidences": [{"user": {"name": "[USER_01]"}}],
    }

    tokenized = tokenize_log(original, key_ring, connection, "INC-001")
    assert "[HOST_01]" not in json.dumps(tokenized)
    assert "logs[6124]" in tokenized["metadata"]["log_source"]

    restored = detokenize_log(tokenized, connection, key_ring, "INC-001", "analyst")
    assert restored == original
    assert [outcome for *_, outcome in connection.audit_log] == ["success"] * 4


def test_invented_placeholder_inside_a_document_is_refused(connection, key_ring):
    tokenized = tokenize_log({"host": "[HOST_01]"}, key_ring, connection, "INC-001")
    tokenized["host"] = "[HOST_99]"

    with pytest.raises(TokenHallucinationError):
        detokenize_log(tokenized, connection, key_ring, "INC-001", "analyst")

    assert connection.audit_log == [("[HOST_99]", "INC-001", "analyst", "rejected")]

## Key rotation

In [ ]:
def test_token_issued_before_rotation_still_decrypts(connection, env):
    """The whole point of the ring: rotating must not orphan old tokens."""
    token = tokenize_log("[HOST_01]", load_key_ring(), connection, "INC-001")

    ring = rotate_to_v2(env)
    assert ring.active_version == "v2"
    assert ring.versions == ("v1", "v2")

    assert safe_decrypt(connection, ring, token, "INC-001", "analyst") == "[HOST_01]"
    assert connection.audit_log[-1] == (token, "INC-001", "analyst", "success")


def test_rotation_changes_the_token_issued_for_the_same_value(connection, env):
    before = tokenize_log("[HOST_01]", load_key_ring(), connection, "INC-001")
    after = tokenize_log("[HOST_01]", rotate_to_v2(env), connection, "INC-002")

    assert before != after
    assert connection.issued[(before, "INC-001")][2] == "v1"
    assert connection.issued[(after, "INC-002")][2] == "v2"


def test_one_log_can_mix_key_versions(connection, env):
    old = tokenize_log("[HOST_01]", load_key_ring(), connection, "INC-001")
    ring = rotate_to_v2(env)
    new = tokenize_log("[USER_02]", ring, connection, "INC-001")

    document = {"before": old, "after": new}
    restored = detokenize_log(document, connection, ring, "INC-001", "analyst")

    assert restored == {"before": "[HOST_01]", "after": "[USER_02]"}


def test_new_tokens_are_stamped_by_the_key_not_the_environment(connection, rotated, env):
    """The version recorded must describe the key that actually encrypted."""
    env.setenv("FF3_ACTIVE_KEY_VERSION", "v9")  # changed after the ring was built

    tokenize_log("[HOST_01]", rotated, connection, "INC-001")

    assert {version for _, _, version in connection.issued.values()} == {"v2"}


def test_key_taken_off_the_ring_is_refused_and_audited(connection, env):
    """The one genuine 'cannot decrypt' case, distinct from merely being old."""
    token = tokenize_log("[HOST_01]", load_key_ring(), connection, "INC-001")

    ring = retire_v1(env)
    assert ring.versions == ("v2",)

    with pytest.raises(UnknownKeyVersionError) as error:
        safe_decrypt(connection, ring, token, "INC-001", "analyst")

    assert error.value.version == "v1"
    assert error.value.token == token
    assert error.value.available == ("v2",)
    assert connection.audit_log[-1] == (token, "INC-001", "analyst", "rejected")


def test_key_versions_in_use_reports_what_blocks_retirement(connection, env):
    tokenize_log({"a": "[HOST_01]", "b": "[USER_02]"}, load_key_ring(), connection, "INC-001")
    tokenize_log("[HOST_03]", rotate_to_v2(env), connection, "INC-002")

    assert key_versions_in_use(connection) == {"v1": 2, "v2": 1}
    assert key_versions_in_use(connection, "INC-001") == {"v1": 2}
    assert key_versions_in_use(connection, "INC-002") == {"v2": 1}


def test_key_ring_resolves_versions_case_insensitively(rotated):
    assert rotated.for_version("V1") is rotated.for_version("v1")
    assert rotated.for_version("v1").version == "v1"


def test_active_version_must_be_on_the_ring(key_ring):
    with pytest.raises(RuntimeError, match="not on the key ring"):
        KeyRing([key_ring.active], "v9")


def test_ring_is_indexed_by_the_version_each_key_carries(key_ring):
    """The provider seam: any source of VersionedCipher works, and the name a
    token is stamped with is the name the ring is searched by."""
    ring = KeyRing([VersionedCipher("KMS-2026-01", key_ring.active.cipher)], "kms-2026-01")

    assert ring.versions == ("kms-2026-01",)
    assert ring.active.version == "kms-2026-01"
    assert ring.for_version("KMS-2026-01") is ring.active


def test_a_provider_name_survives_issuance_and_detokenization(connection, key_ring):
    """End to end for the seam: whatever a provider calls a key, the name the
    token is stamped with and the name the ring resolves must stay the same."""
    ring = KeyRing([VersionedCipher("KMS-2026-01", key_ring.active.cipher)], "KMS-2026-01")

    token = tokenize_log("[HOST_01]", ring, connection, "INC-001")

    assert connection.issued[(token, "INC-001")][2] == "kms-2026-01"
    assert safe_decrypt(connection, ring, token, "INC-001", "analyst") == "[HOST_01]"


def test_key_material_is_never_rendered(key_ring):
    """Keys must not leak into logs or tracebacks through repr()."""
    assert KEY_V1 not in repr(key_ring)
    assert KEY_V1 not in repr(key_ring.active)
    assert TWEAK_V1 not in repr(key_ring)

## Configuration

In [ ]:
def test_environment_variables_are_required(env):
    env.delenv("FF3_KEY")
    env.delenv("FF3_TWEAK")

    with pytest.raises(RuntimeError, match="FF3_KEY"):
        load_key_ring()

    env.setenv("FF3_KEY", "key")
    with pytest.raises(RuntimeError, match="FF3_TWEAK"):
        load_key_ring()


def test_key_version_env_name_is_not_mistaken_for_a_key(env):
    """FF3_KEY_VERSION matches the FF3_KEY_<V> pattern but is not a key."""
    env.setenv("FF3_KEY_VERSION", "v7")

    ring = load_key_ring()

    assert ring.versions == ("v7",)
    assert "version" not in ring.versions


def test_database_url_is_required(monkeypatch):
    monkeypatch.delenv("DATABASE_URL", raising=False)

    with pytest.raises(RuntimeError, match="DATABASE_URL"):
        namespace["connect_database"]()


@pytest.mark.parametrize(
    "value", ["[HOST_01]", "[INT_IP_01]", "[LOG_COLLECTOR_01]", "[HOST_A_B]"]
)
def test_pad_character_never_appears_in_a_captured_suffix(value):
    """The padding scheme is only reversible while this holds.

    A '_' inside a placeholder is always absorbed into the prefix, never into
    the suffix, so padding can never be confused with real content.
    """
    assert PAD_CHAR in ALPHABET
    match = namespace["PLACEHOLDER_RE"].fullmatch(value)
    assert match is not None
    assert PAD_CHAR not in match.group(2)


def test_schema_is_idempotent_and_covers_both_tables(connection):
    schema_sql = namespace["SCHEMA_SQL"]

    assert "CREATE TABLE IF NOT EXISTS vault_schema.issued_tokens" in schema_sql
    assert "CREATE TABLE IF NOT EXISTS metadata_schema.detokenize_log" in schema_sql
    assert "CREATE SCHEMA IF NOT EXISTS vault_schema" in schema_sql
    assert "CREATE SCHEMA IF NOT EXISTS metadata_schema" in schema_sql
    assert "UNIQUE (token, case_id)" in schema_sql
    assert "key_version TEXT NOT NULL" in schema_sql

    ensure_schema(connection)
    assert connection.schema_ready


def test_issued_token_insert_is_idempotent(connection):
    record_issued_token(connection, "[HOST_ABC]", "HOST", 2, "INC-001", "v1")
    record_issued_token(connection, "[HOST_ABC]", "HOST", 2, "INC-001", "v1")

    insert_queries = [query for query, _ in connection.executed]
    assert insert_queries[0].startswith("INSERT INTO VAULT_SCHEMA.ISSUED_TOKENS")
    assert "ON CONFLICT (TOKEN, CASE_ID) DO NOTHING" in insert_queries[0]
    assert len(connection.issued) == 1
    assert lookup_issued_token(connection, "[HOST_ABC]", "INC-001") == ("HOST", 2, "v1")

## Run the tests

The cell below gathers every cell tagged `test` into a temporary module
and runs real pytest over it, so fixtures, `parametrize` and assertion
rewriting all behave normally. **Save the notebook first** - the runner
reads this file from disk, not the live kernel.

In [ ]:
import subprocess
import sys
import tempfile


def build_test_module() -> str:
    """Concatenate the cells of this notebook that are tagged 'test'."""
    this_notebook = locate_notebook("test_encryp.ipynb")
    with this_notebook.open(encoding="utf-8") as notebook_file:
        cells = json.load(notebook_file)["cells"]
    tagged = [
        cell
        for cell in cells
        if cell["cell_type"] == "code" and "test" in cell["metadata"].get("tags", [])
    ]
    if not tagged:
        raise AssertionError("no code cell in test_encryp.ipynb is tagged 'test'")
    return "\n\n".join("".join(cell["source"]) for cell in tagged)


def run_tests(*pytest_args: str) -> int:
    """Run the tagged cells under real pytest and print its report.

    Reads this notebook from disk, so save it before running. Pass extra
    pytest arguments through, e.g. run_tests("-k", "round_trip", "-v").
    """
    with tempfile.TemporaryDirectory() as directory:
        module_path = Path(directory) / "test_encryp_cells.py"
        module_path.write_text(build_test_module(), encoding="utf-8")
        completed = subprocess.run(
            [sys.executable, "-m", "pytest", str(module_path), "-q", *pytest_args],
            capture_output=True,
            text=True,
            env={**os.environ, "ENCRYP_NOTEBOOK": str(NOTEBOOK.resolve())},
        )
    print(completed.stdout or completed.stderr)
    return completed.returncode


run_tests()